# Reading `temporal_pack_compact_clean.h5`

This notebook explains how the compact pack file is structured and how to read it,
including its two HDF5 keys, the columns in each, and how to join them back together.

---

## Requirements

- Python with **`pandas`**, **`pytables`**, and **`numpy >= 2.0`**.
- The activity-trace arrays were pickled under **numpy 2.x**. Reading them under
  numpy 1.x raises `ModuleNotFoundError`. Run the version check below first.


In [1]:
import numpy as np
import pandas as pd

print('numpy :', np.__version__, '  (need >= 2.0 to unpickle the trace arrays)')
print('pandas:', pd.__version__)
assert int(np.__version__.split('.')[0]) >= 2, 'Use a numpy>=2.0 environment to read the traces.'


numpy : 2.4.6   (need >= 2.0 to unpickle the trace arrays)
pandas: 2.2.3


## Set the path to the file

Point this at wherever the file lives on your machine.


In [2]:
from pathlib import Path

path = Path('/Volumes/Kobi/DOI_animals_good/temporal_pack_compact_clean.h5')
assert path.is_file(), f'File not found: {path}'
print('OK:', path, f'({path.stat().st_size/1e9:.2f} GB)')


OK: /Volumes/Kobi/DOI_animals_good/temporal_pack_compact_clean.h5 (1.71 GB)


## The file has TWO keys

Always pass `key=...` explicitly. A bare `pd.read_hdf(path)` will error because
there is more than one key in the file.

| Key | What it is | Row identity |
|-----|-----------|--------------|
| `/compact_temporal` | Main table: one row per ROI, with the activity traces | the DataFrame index |
| `/com_aligned` | Side table: atlas-alignment results per ROI | `pack_index` column = the `/compact_temporal` index |


In [3]:
with pd.HDFStore(path, 'r') as st:
    print('keys:', list(st.keys()))


keys: ['/com_aligned', '/compact_temporal']


## `/compact_temporal` — the main trace table

One row per ROI. Columns:

- **`fish_id`, `plane`, `neuron`, `roi_index`** — identifiers
- **`com`** — ROI center of mass in raw STD coords `(col, row)`
- **`raw_norm_temporal`, `norm_temporal`** — activity trace arrays (numpy)
- **`pulse_frames`, `age`, `concentration`** — experiment metadata


In [4]:
base = pd.read_hdf(path, key='compact_temporal')
print('rows:', len(base))
print('columns:', list(base.columns))
base.head(3)


rows: 121950
columns: ['fish_id', 'plane', 'neuron', 'raw_norm_temporal', 'norm_temporal', 'roi_index', 'com', 'pulse_frames', 'age', 'concentration']


,fish_id,plane,neuron,raw_norm_temporal,norm_temporal,roi_index,com,pulse_frames,age,concentration
0,7,0,0,"[0.26243988, 0.3913488, 0.422058, 0.30056942, ...","[0.0, 0.2655894, 0.17940561, 0.12118849, 0.081...",5,"[246.0624376037795, 206.42705475831755]",[292],8,0.0
1,7,0,1,"[1.0, 0.4782451, 0.72655195, 0.5480897, 0.4895...","[1.0, 0.86915547, 0.7586579, 0.65939164, 0.573...",19,"[112.83796864247208, 233.1113084660113]",[292],8,0.0
2,7,0,2,"[0.5393721, 0.31377077, 0.39671123, 0.21532322...","[0.19080034, 0.14736302, 0.11381458, 0.0879037...",20,"[106.1070536414167, 247.30094418638612]",[292],8,0.0


## `/com_aligned` — the alignment side table

One row per aligned ROI. Columns:

- **`pack_index`** — join key; equals the `/compact_temporal` index
- **`fish_id`, `plane`, `neuron`, `roi_index`** — identifiers (mirror of the main table)
- **`com_aligned_mapzebrain`** — ROI COM mapped into the mapZebrain atlas frame `(x, y)`
- **`atlas_voxel_mapzebrain`** — 3D atlas voxel `(ax, ay, az)` *(present only for voxel-based fish; `NaN` otherwise)*
- **`region_aligned_mapzebrain`** — list of region labels the ROI falls in
- **`single_region`** — the single priority region label


In [5]:
side = pd.read_hdf(path, key='com_aligned')
print('rows:', len(side))
print('columns:', list(side.columns))
print('\nrows per fish:')
print(side['fish_id'].value_counts().sort_index().to_string())
side.head(3)


rows: 121950
columns: ['pack_index', 'fish_id', 'plane', 'neuron', 'roi_index', 'com_aligned_mapzebrain', 'atlas_voxel_mapzebrain', 'region_aligned_mapzebrain', 'single_region']

rows per fish:
fish_id
7      4244
8      8230
9      8382
10     9725
11     6336
13     8057
15     8754
17     6902
20     7691
21     6323
24     8760
28     9171
30     6553
31    10243
32    12579


,pack_index,fish_id,plane,neuron,roi_index,com_aligned_mapzebrain,atlas_voxel_mapzebrain,region_aligned_mapzebrain,single_region
0,109371,32,0,0,1,"(72.0, 454.0)","(93.26009457471719, 459.48769749163677, 271.67...",[],None
1,109372,32,0,1,2,"(73.0, 439.0)","(94.40147401434986, 447.9478968917525, 270.042...",[],None
2,109373,32,0,2,3,"(69.0, 446.0)","(91.13523102183571, 453.3590823693101, 270.809...",[],None


## Join the alignment back onto the main table

The side table joins on **`pack_index`**, which equals the `/compact_temporal` index.
After this, `base` has traces + alignment + regions all in one DataFrame.


In [ ]:
side_ix = pd.read_hdf(path, key='com_aligned').set_index('pack_index')
for c in ['com_aligned_mapzebrain', 'atlas_voxel_mapzebrain',
          'region_aligned_mapzebrain', 'single_region']:
    if c in side_ix.columns:
        base[c] = side_ix[c]

print('columns now:', list(base.columns))
base.head(3)


## Filter to one fish


In [ ]:
fish_id = 21
one_fish = base[base['fish_id'] == fish_id]
print(f'fish {fish_id}: {len(one_fish)} ROIs')
one_fish.head(3)


## Notes / gotchas

- **Two keys, always specify `key=`.** `/compact_temporal` for traces, `/com_aligned` for alignment.
- **numpy >= 2.0 required** to unpickle `raw_norm_temporal` / `norm_temporal`.
- **Not every fish is aligned.** A fish appears in `/com_aligned` only after its alignment
  notebook has been run. `atlas_voxel_mapzebrain` is populated only for voxel-based fish;
  it is `NaN` for fish aligned with a single 2D reference-plane transform.
- **`region_aligned_mapzebrain` / `single_region`** are filled only if the region-assignment
  step was run before persisting; otherwise they are empty (`[]` / `None`).
